In [2]:
import numpy as np
import pandas as pd

# **Concatenating `pd.DataFrame` objects**

In [3]:
df_q1 = pd.DataFrame([
    ["AAPL", 100., 50., 75.],
    ["MSFT", 80., 42., 62.],
    ["AMZN", 60., 100., 120.],
], columns=["ticker", "shares", "low", "high"]).convert_dtypes(dtype_backend="numpy_nullable")

df_q1

,ticker,shares,low,high
0,AAPL,100,50,75
1,MSFT,80,42,62
2,AMZN,60,100,120


In [4]:
df_q2 = pd.DataFrame([
    ["AAPL", 80., 70., 80., 77.],
    ["MSFT", 90., 50., 60., 55.],
    ["IBM", 100., 60., 70., 64.],
    ["GE", 42., 30., 50., 44.],
], columns=["ticker", "shares", "low", "high", "close"]).convert_dtypes(dtype_backend="numpy_nullable")

df_q2

,ticker,shares,low,high,close
0,AAPL,80,70,80,77
1,MSFT,90,50,60,55
2,IBM,100,60,70,64
3,GE,42,30,50,44


In [6]:
pd.concat([df_q1, df_q2]) # concate method by defualt make the concatenation vertically

,ticker,shares,low,high,close
0,AAPL,100,50,75,<NA>
1,MSFT,80,42,62,<NA>
2,AMZN,60,100,120,<NA>
0,AAPL,80,70,80,77
1,MSFT,90,50,60,55
2,IBM,100,60,70,64
3,GE,42,30,50,44


In [8]:
pd.concat(
    [df_q1, df_q2],
    ignore_index=True
)

,ticker,shares,low,high,close
0,AAPL,100,50,75,<NA>
1,MSFT,80,42,62,<NA>
2,AMZN,60,100,120,<NA>
3,AAPL,80,70,80,77
4,MSFT,90,50,60,55
5,IBM,100,60,70,64
6,GE,42,30,50,44


In [10]:
pd.concat(
    [df_q1, df_q2],
    # ignore_index=True,
    keys=["q1", "q2"]
)

ticker  shares  low  high  close
q1 0   AAPL     100   50    75   <NA>
   1   MSFT      80   42    62   <NA>
   2   AMZN      60  100   120   <NA>
q2 0   AAPL      80   70    80     77
   1   MSFT      90   50    60     55
   2    IBM     100   60    70     64
   3     GE      42   30    50     44

In [13]:
pd.concat(
    [df_q1, df_q2],
    keys=["q1", "q2"],
    axis=1
)

q1                        q2                      
  ticker shares   low  high ticker shares low high close
0   AAPL    100    50    75   AAPL     80  70   80    77
1   MSFT     80    42    62   MSFT     90  50   60    55
2   AMZN     60   100   120    IBM    100  60   70    64
3   <NA>   <NA>  <NA>  <NA>     GE     42  30   50    44

In [14]:
pd.concat(
    [
        df_q1.set_index("ticker"), 
        df_q2.set_index("ticker")
    ],
    keys=["q1", "q2"],
    axis=1
)

q1                 q2                  
       shares   low  high shares   low  high close
ticker                                            
AAPL      100    50    75     80    70    80    77
MSFT       80    42    62     90    50    60    55
AMZN       60   100   120   <NA>  <NA>  <NA>  <NA>
IBM      <NA>  <NA>  <NA>    100    60    70    64
GE       <NA>  <NA>  <NA>     42    30    50    44

In [15]:
pd.concat(
    [
        df_q1.set_index("ticker"), 
        df_q2.set_index("ticker")
    ],
    keys=["q1", "q2"],
    axis=1,
    join='inner'
)

q1              q2               
       shares low high shares low high close
ticker                                      
AAPL      100  50   75     80  70   80    77
MSFT       80  42   62     90  50   60    55

> **`pd.concat`** is an expensive operation, and should never be called from within a Python loop.

In [16]:
%%time
concatenated_dfs = df_q1

for i in range(1_000):
    concatenated_dfs = pd.concat([concatenated_dfs, df_q1])

print(f"Final pd.DataFrame shape is {concatenated_dfs.shape}")

Final pd.DataFrame shape is (3003, 4)
CPU times: user 388 ms, sys: 0 ns, total: 388 ms
Wall time: 388 ms


In [19]:
%%time

df = df_q1
accumulated = [df_q1]

for i in range(1000):
    accumulated.append(df_q1)

concatenated_dfs = pd.concat(accumulated)

print(f"Final pd.DataFrame shape is {concatenated_dfs.shape}")

Final pd.DataFrame shape is (3003, 4)
CPU times: user 85.6 ms, sys: 3.19 ms, total: 88.8 ms
Wall time: 87.1 ms


# **Merging DataFrames with `pd.merge`**

In [23]:
df_q1 = pd.DataFrame([
    ["AAPL", 100., 50., 75.],
    ["MSFT", 80., 42., 62.],
    ["AMZN", 60., 100., 120.],
], columns=["ticker", "shares", "low", "high"]).convert_dtypes(dtype_backend="numpy_nullable")

df_q1

,ticker,shares,low,high
0,AAPL,100,50,75
1,MSFT,80,42,62
2,AMZN,60,100,120


In [25]:
df_q2 = pd.DataFrame([
    ["AAPL", 80., 70., 80., 77.],
    ["MSFT", 90., 50., 60., 55.],
    ["IBM", 100., 60., 70., 64.],
    ["GE", 42., 30., 50., 44.],
], columns=["ticker", "shares", "low", "high", "close"]).convert_dtypes(dtype_backend="numpy_nullable")

df_q2

,ticker,shares,low,high,close
0,AAPL,80,70,80,77
1,MSFT,90,50,60,55
2,IBM,100,60,70,64
3,GE,42,30,50,44


In [26]:
pd.concat(
    [
    df_q1.set_index("ticker"),
    df_q2.set_index("ticker")
    ],
    keys=["q1", "q2"], 
    axis=1)

q1                 q2                  
       shares   low  high shares   low  high close
ticker                                            
AAPL      100    50    75     80    70    80    77
MSFT       80    42    62     90    50    60    55
AMZN       60   100   120   <NA>  <NA>  <NA>  <NA>
IBM      <NA>  <NA>  <NA>    100    60    70    64
GE       <NA>  <NA>  <NA>     42    30    50    44

In [28]:
df_q1.merge(
    df_q2,
    on=["ticker"]
)

,ticker,shares_x,low_x,high_x,shares_y,low_y,high_y,close
0,AAPL,100,50,75,80,70,80,77
1,MSFT,80,42,62,90,50,60,55


In [29]:
pd.merge(
    df_q1, 
    df_q2, 
    on=["ticker"], 
    how="outer"
)

,ticker,shares_x,low_x,high_x,shares_y,low_y,high_y,close
0,AAPL,100,50,75,80,70,80,77
1,AMZN,60,100,120,<NA>,<NA>,<NA>,<NA>
2,GE,<NA>,<NA>,<NA>,42,30,50,44
3,IBM,<NA>,<NA>,<NA>,100,60,70,64
4,MSFT,80,42,62,90,50,60,55


In [30]:
pd.merge(
    df_q1, 
    df_q2, 
    on=["ticker"], 
    how="left"
)

,ticker,shares_x,low_x,high_x,shares_y,low_y,high_y,close
0,AAPL,100,50,75,80,70,80,77
1,MSFT,80,42,62,90,50,60,55
2,AMZN,60,100,120,<NA>,<NA>,<NA>,<NA>


In [31]:
pd.merge(
    df_q1, 
    df_q2, 
    on=["ticker"], 
    how="right"
)

,ticker,shares_x,low_x,high_x,shares_y,low_y,high_y,close
0,AAPL,100,50,75,80,70,80,77
1,MSFT,80,42,62,90,50,60,55
2,IBM,<NA>,<NA>,<NA>,100,60,70,64
3,GE,<NA>,<NA>,<NA>,42,30,50,44


In [32]:
pd.merge(
    df_q1, 
    df_q2, 
    on=["ticker"], 
    how="outer", 
    indicator=True
)

,ticker,shares_x,low_x,high_x,shares_y,low_y,high_y,close,_merge
0,AAPL,100,50,75,80,70,80,77,both
1,AMZN,60,100,120,<NA>,<NA>,<NA>,<NA>,left_only
2,GE,<NA>,<NA>,<NA>,42,30,50,44,right_only
3,IBM,<NA>,<NA>,<NA>,100,60,70,64,right_only
4,MSFT,80,42,62,90,50,60,55,both


In [33]:
pd.merge(
    df_q1,
    df_q2,
    on=["ticker"],
    how="outer",
    suffixes=("_q1", "_q2"),
)

,ticker,shares_q1,low_q1,high_q1,shares_q2,low_q2,high_q2,close
0,AAPL,100,50,75,80,70,80,77
1,AMZN,60,100,120,<NA>,<NA>,<NA>,<NA>
2,GE,<NA>,<NA>,<NA>,42,30,50,44
3,IBM,<NA>,<NA>,<NA>,100,60,70,64
4,MSFT,80,42,62,90,50,60,55


In [34]:
pd.merge(
    df_q1[["ticker"]].assign(only_in_left=42),
    df_q2[["ticker"]].assign(only_in_right=555),
    on=["ticker"],
    how="outer",
    suffixes=("_q1", "_q2"),
)

,ticker,only_in_left,only_in_right
0,AAPL,42.0,555.0
1,AMZN,42.0,NaN
2,GE,NaN,555.0
3,IBM,NaN,555.0
4,MSFT,42.0,555.0


In [37]:
df_q2 = df_q2.rename(columns={"ticker": "SYMBOL"})

df_q2

,SYMBOL,shares,low,high,close
0,AAPL,80,70,80,77
1,MSFT,90,50,60,55
2,IBM,100,60,70,64
3,GE,42,30,50,44


In [38]:
pd.merge(
    df_q1,
    df_q2,
    left_on=["ticker"],
    right_on=["SYMBOL"],
    how="outer",
    suffixes=("_q1", "_q2"),
)

,ticker,shares_q1,low_q1,high_q1,SYMBOL,shares_q2,low_q2,high_q2,close
0,AAPL,100,50,75,AAPL,80,70,80,77
1,AMZN,60,100,120,<NA>,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>,GE,42,30,50,44
3,<NA>,<NA>,<NA>,<NA>,IBM,100,60,70,64
4,MSFT,80,42,62,MSFT,90,50,60,55


In [39]:
lows = pd.DataFrame([
    ["AAPL", "Q1", 50.],
    ["MSFT", "Q1", 42.],
    ["AMZN", "Q1", 100.],
    ["AAPL", "Q2", 70.],
    ["MSFT", "Q2", 50.],
    ["IBM", "Q2", 60.],
    ["GE", "Q2", 30.],
], columns=["ticker", "quarter", "low"]).convert_dtypes(dtype_backend="numpy_nullable")

lows

,ticker,quarter,low
0,AAPL,Q1,50
1,MSFT,Q1,42
2,AMZN,Q1,100
3,AAPL,Q2,70
4,MSFT,Q2,50
5,IBM,Q2,60
6,GE,Q2,30


In [40]:
highs = pd.DataFrame([
    ["AAPL", "Q1", 75.],
    ["MSFT", "Q1", 62.],
    ["AMZN", "Q1", 120.],
    ["AAPL", "Q2", 80.],
    ["MSFT", "Q2", 60.],
    ["IBM", "Q2", 70.],
    ["GE", "Q2", 50.],
], columns=["SYMBOL", "QTR", "high"]).convert_dtypes(dtype_backend="numpy_nullable")

highs

,SYMBOL,QTR,high
0,AAPL,Q1,75
1,MSFT,Q1,62
2,AMZN,Q1,120
3,AAPL,Q2,80
4,MSFT,Q2,60
5,IBM,Q2,70
6,GE,Q2,50


In [41]:
pd.merge(
    lows,
    highs,
    left_on=["ticker", "quarter"],
    right_on=["SYMBOL", "QTR"]
)

,ticker,quarter,low,SYMBOL,QTR,high
0,AAPL,Q1,50,AAPL,Q1,75
1,MSFT,Q1,42,MSFT,Q1,62
2,AMZN,Q1,100,AMZN,Q1,120
3,AAPL,Q2,70,AAPL,Q2,80
4,MSFT,Q2,50,MSFT,Q2,60
5,IBM,Q2,60,IBM,Q2,70
6,GE,Q2,30,GE,Q2,50


In [42]:
sales = pd.DataFrame([
    ["Jan", "John", 10],
    ["Feb", "John", 20],
    ["Mar", "John", 30],
], columns=["month", "salesperson", "sales"]).convert_dtypes(dtype_backend="numpy_nullable")

sales

,month,salesperson,sales
0,Jan,John,10
1,Feb,John,20
2,Mar,John,30


In [43]:
regions = pd.DataFrame([
    ["John", "Northeast"],
    ["Jane", "Southwest"],
], columns=["salesperson", "region"]).convert_dtypes(dtype_backend="numpy_nullable")

regions

,salesperson,region
0,John,Northeast
1,Jane,Southwest


In [44]:
pd.merge(sales, regions, on=["salesperson"])

,month,salesperson,sales,region
0,Jan,John,10,Northeast
1,Feb,John,20,Northeast
2,Mar,John,30,Northeast


In [45]:
pd.merge(sales, regions, on=["salesperson"])["sales"].sum()

np.int64(60)

In [46]:
regions_orig = regions

regions = pd.DataFrame([
    ["John", "Smith", "Northeast"],
    ["Jane", "Doe", "Southwest"],
    ["John", "Newhire", "Southeast"],
], columns=["salesperson", "last_name", "region"]).convert_dtypes(dtype_backend="numpy_nullable")

regions

,salesperson,last_name,region
0,John,Smith,Northeast
1,Jane,Doe,Southwest
2,John,Newhire,Southeast


In [47]:
pd.merge(sales, regions, on=["salesperson"])

,month,salesperson,sales,last_name,region
0,Jan,John,10,Smith,Northeast
1,Jan,John,10,Newhire,Southeast
2,Feb,John,20,Smith,Northeast
3,Feb,John,20,Newhire,Southeast
4,Mar,John,30,Smith,Northeast
5,Mar,John,30,Newhire,Southeast


In [48]:
pd.merge(sales, regions, on=["salesperson"])["sales"].sum()

np.int64(120)

In [56]:
pd.merge(sales, regions_orig, on=["salesperson"], validate="many_to_one")

,month,salesperson,sales,region
0,Jan,John,10,Northeast
1,Feb,John,20,Northeast
2,Mar,John,30,Northeast


In [58]:
# pd.merge(sales, regions, on=["salesperson"], validate="many_to_one") => MergeError: Merge keys are not unique in right dataset; 
#    not a many-to-one merge

# **Joining DataFrames with `pd.DataFrame.join`**

In [59]:
sales = pd.DataFrame(
    [[1000], [2000], [4000]],
    columns=["sales"],
    index=pd.Index([42, 555, 9000], name="salesperson_id")).convert_dtypes(dtype_backend="numpy_nullable")

sales

,sales
salesperson_id,
42,1000
555,2000
9000,4000


In [60]:
salesperson = pd.DataFrame([
    ["John", "Smith"],
    ["Jane", "Doe"],
    ], 
    columns=["first_name", "last_name"], 
    index=pd.Index([555, 42], name="salesperson_id")).convert_dtypes(dtype_backend="numpy_nullable")

salesperson

,first_name,last_name
salesperson_id,,
555,John,Smith
42,Jane,Doe


In [61]:
pd.merge(
    sales, 
    salesperson, 
    left_index=True, 
    right_index=True,
    how="left"
)

,sales,first_name,last_name
salesperson_id,,,
42,1000,Jane,Doe
555,2000,John,Smith
9000,4000,<NA>,<NA>


In [62]:
sales.join(salesperson)

,sales,first_name,last_name
salesperson_id,,,
42,1000,Jane,Doe
555,2000,John,Smith
9000,4000,<NA>,<NA>


In [63]:
sales.join(salesperson, how="inner")

,sales,first_name,last_name
salesperson_id,,,
42,1000,Jane,Doe
555,2000,John,Smith


# **Reshaping with `pd.DataFrame.stack` and `pd.DataFrame.unstack`**